**1) Data Types & Missing Values**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import csv
from io import StringIO

### Load Dataset

In [ ]:
url = "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/main/healthcare-disease-prediction/dataset/health_dataset.csv"

# Download dataset from GitHub
response = requests.get(url)
response.raise_for_status()

# Read CSV while preserving rows with extra fields
rows = list(csv.reader(StringIO(response.text)))

header = rows[0]
data = rows[1:]

# Find the maximum number of fields in any row
max_fields = max(len(row) for row in data)

# Add extra symptom columns if required
for i in range(len(header), max_fields):
    header.append(f"Symptom_{i}")

# Make every row have the same number of fields
data = [row + [""] * (len(header) - len(row)) for row in data]

df = pd.DataFrame(data, columns=header)

print("Dataset loaded successfully.")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

In [ ]:
print("Dataset Shape:", df.shape)

In [ ]:
print("First 5 rows:")
display(df.head())

In [ ]:
print("Last 5 rows:")
display(df.tail())

In [ ]:
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

In [ ]:
print("Column Names:")
print(df.columns.tolist())

In [ ]:
print("Dataset Information:")
df.info()

In [ ]:
dtype_df = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

display(dtype_df)

In [ ]:
print("Data Type Counts:")
print(df.dtypes.value_counts())

In [ ]:
print("Categorical / Text Columns:")
categorical_columns = df.select_dtypes(include="object").columns.tolist()
print(categorical_columns)

### Unique Values

In [ ]:
print("Number of Unique Diseases:", df["Disease"].nunique())

print("\nDisease Names:")
print(df["Disease"].unique())

In [ ]:
print("Unique Value Counts for Each Column:")

unique_counts = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": [df[col].nunique(dropna=True) for col in df.columns]
})

display(unique_counts)

### Disease Counts

In [ ]:
disease_counts = df["Disease"].value_counts()

print("Disease Counts:")
display(disease_counts.to_frame("Count"))

In [ ]:
plt.figure(figsize=(12, 6))
disease_counts.plot(kind="bar")
plt.title("Number of Records per Disease")
plt.xlabel("Disease")
plt.ylabel("Number of Records")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

### Symptom Columns

In [ ]:
symptom_columns = [col for col in df.columns if col.startswith("Symptom_")]

print("Symptom Columns:")
print(symptom_columns)
print("Total Symptom Columns:", len(symptom_columns))

### Missing Values

In [ ]:
# Convert empty and whitespace-only values to NaN
df = df.replace(r"^\s*$", np.nan, regex=True)

print("Missing Values:")
display(df.isnull().sum().to_frame("Missing Values"))

In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": missing_percentage.round(2)
})

display(missing_df)

In [ ]:
print("Total Missing Values:", df.isnull().sum().sum())

### Missing Values in Symptoms

In [ ]:
symptom_missing = pd.DataFrame({
    "Missing Values": df[symptom_columns].isnull().sum(),
    "Missing Percentage": (df[symptom_columns].isnull().sum() / len(df) * 100).round(2)
})

display(symptom_missing)

### Available Symptoms per Record

In [ ]:
df["Available_Symptoms"] = df[symptom_columns].notna().sum(axis=1)

print("Available Symptoms per Record:")
display(df[["Disease", "Available_Symptoms"]].head(10))

In [ ]:
print("Distribution of Available Symptoms:")
display(df["Available_Symptoms"].value_counts().sort_index().to_frame("Number of Records"))

### Incomplete Records

In [ ]:
incomplete_records = df[df["Available_Symptoms"] < len(symptom_columns)]

print("Number of incomplete records:", len(incomplete_records))
display(incomplete_records.head(10))

### Records with Additional Symptoms

In [ ]:
if "Symptom_13" in df.columns:
    extra_symptom_records = df[df["Symptom_13"].notna()]
    print("Records containing Symptom_13:", len(extra_symptom_records))
    display(extra_symptom_records[["Disease", "Symptom_13"]].head(10))
else:
    print("No Symptom_13 column found.")

### Complete Records

In [ ]:
complete_records = df[df["Available_Symptoms"] >= 12]

print("Number of records with at least 12 symptoms:", len(complete_records))

### Data Quality Summary

In [ ]:
print("========== DATA QUALITY SUMMARY ==========")
print("Total Records:", len(df))
print("Total Columns:", len(df.columns))
print("Unique Diseases:", df["Disease"].nunique())
print("Total Missing Values:", df.isnull().sum().sum())
print("Incomplete Records:", len(incomplete_records))

if "Symptom_13" in df.columns:
    print("Records with Symptom_13:", df["Symptom_13"].notna().sum())

### Summary

- The dataset was loaded directly from the GitHub repository.
- Rows containing extra symptom fields were preserved instead of being skipped.
- Additional symptom columns were created automatically when required.
- Dataset shape, column names and data types were examined.
- Unique diseases and disease frequencies were calculated.
- Missing and blank values were identified.
- Missing values in individual symptom columns were analyzed.
- The number of available symptoms for each record was calculated.
- Incomplete records were identified without deleting them.
- The dataset was checked for basic data-quality issues.